In [1]:
import snowflake.connector
import pandas as pd
import os
from dotenv import load_dotenv

In [2]:
load_dotenv("/home/scooperlosses/DB_proj/snowflake_dbt_ETL/etl_proj/.env")


con = snowflake.connector.connect(
    account=os.getenv("SHOWFLAKE_ACCOUNT"),
    user=os.getenv("SHOWFLAKE_USER"),
    password=os.getenv("SHOWFLAKE_PASSWORD"),
    database=os.getenv("DATABASE"),
    schema="SILVER",
    warehouse=os.getenv("WAREHOUSE")
)


cursor = con.cursor()
cursor.execute("SELECT * FROM CAFEDB._DEV_SILVER.int_clean_rest")
df = cursor.fetch_pandas_all()
df.head()

,TRANSACTION_ID,ITEMS,PRICE_PER_UNIT,QUANTITIES,TOTAL_SPENT,PAYMENT_METHOD,LOCATION,TRANSACTION_DATE
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,3.0,4.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,1.0,4.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,5.0,2.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9324 entries, 0 to 9323
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   TRANSACTION_ID    9324 non-null   object 
 1   ITEMS             9324 non-null   object 
 2   PRICE_PER_UNIT    9324 non-null   float64
 3   QUANTITIES        9324 non-null   float64
 4   TOTAL_SPENT       9324 non-null   float64
 5   PAYMENT_METHOD    9324 non-null   object 
 6   LOCATION          9324 non-null   object 
 7   TRANSACTION_DATE  9324 non-null   object 
dtypes: float64(3), object(5)
memory usage: 582.9+ KB


In [3]:
df.columns

Index(['TRANSACTION_ID', 'ITEMS', 'PRICE_PER_UNIT', 'QUANTITIES',
       'TOTAL_SPENT', 'PAYMENT_METHOD', 'LOCATION', 'TRANSACTION_DATE'],
      dtype='object')

In [18]:
df[['QUANTITIES','PRICE_PER_UNIT', 'TOTAL_SPENT']]

,QUANTITIES,PRICE_PER_UNIT,TOTAL_SPENT
0,2.0,2.0,4.0
1,4.0,3.0,12.0
2,4.0,1.0,4.0
3,2.0,5.0,10.0
4,2.0,2.0,4.0
...,...,...,...
9319,2.0,4.0,8.0
9320,2.0,2.0,4.0
9321,4.0,2.0,8.0
9322,3.0,1.0,3.0


In [30]:
df.loc[df['QUANTITIES'].isnull(), ['QUANTITIES', 'PRICE_PER_UNIT', 'TOTAL_SPENT']]

,QUANTITIES,PRICE_PER_UNIT,TOTAL_SPENT


In [32]:
df[df.isnull().any(axis=1)].head(10)
#df.isnull().sum()

,TRANSACTION_ID,ITEMS,PRICE_PER_UNIT,QUANTITIES,TOTAL_SPENT,PAYMENT_METHOD,LOCATION,TRANSACTION_DATE
5,TXN_2602893,Smoothie,4.0,5.0,20.0,Credit Card,None,2023-03-31
7,TXN_2064365,Sandwich,4.0,5.0,20.0,None,In-store,2023-12-31
11,TXN_9437049,Cookie,1.0,5.0,5.0,None,Takeaway,2023-06-01
12,TXN_8915701,Tea,1.5,2.0,3.0,None,In-store,2023-03-21
14,TXN_3765707,Sandwich,4.0,1.0,4.0,None,None,2023-06-10
21,TXN_2616390,Sandwich,4.0,2.0,8.0,None,None,2023-09-18
26,TXN_8467949,Smoothie,4.0,5.0,20.0,Credit Card,None,2023-03-11
28,TXN_9677376,Smoothie,4.0,4.0,16.0,None,In-store,2023-08-15
29,TXN_7710508,Cookie,1.0,5.0,5.0,Cash,None,ERROR
33,TXN_2655815,Smoothie,4.0,4.0,16.0,None,Takeaway,2023-06-08


In [31]:
df.groupby('ITEMS', as_index=False)['PRICE_PER_UNIT'].first()

,ITEMS,PRICE_PER_UNIT
0,Cake,3.0
1,Coffee,2.0
2,Cookie,1.0
3,Juice,3.0
4,Salad,5.0
5,Sandwich,4.0
6,Smoothie,4.0
7,Tea,1.5
